# Day 10: Production FastAPI: dependencies, auth, errors, pytest, logging, idempotency (Exercises)

**Beyond Python for Everybody.** Reference: the FastAPI tutorial and advanced user guide (fastapi.tiangolo.com), and the pytest docs (docs.pytest.org).

**Why it matters at ~4 years of experience:** Day 7 showed you can build a route. Here you show you can run a service: dependency injection you can override in tests, auth that returns the right status codes, pagination, consistent errors, real pytest style, logging and config, and safe retries for payments.

**Time:** about 110 minutes · 7 exercises

**How to use this notebook**
1. Run the first cell (it defines `check()`).
2. For each exercise: read the prompt, write your code in the code cell, run it, then run the check cell below it.
3. ✅ means it passed. ❌ prints a hint. Fix and re-run until green.
4. Don't open `solutions.ipynb` until you've tried. If stuck for 10 minutes, read only the hint.
5. Run the last cell for your score, then answer the *Say it out loud* questions.

In [ ]:
# Run this cell first. It gives you the check() helper used below.
_results = {}

def check(name, test):
    """Run a test function and print a pass/fail line with a hint."""
    try:
        test()
    except AssertionError as e:
        _results[name] = False
        print(f"❌ {name}: {e}")
    except Exception as e:
        _results[name] = False
        print(f"❌ {name}: {type(e).__name__}: {e}")
    else:
        _results[name] = True
        print(f"✅ {name}")

def raises(exc, fn, *args, **kwargs):
    """True if fn(*args, **kwargs) raises exc."""
    try:
        fn(*args, **kwargs)
    except exc:
        return True
    except Exception:
        return False
    return False

TOTAL = 7

def score():
    done = sum(_results.values())
    print(f"Score: {done}/{TOTAL}  " + ("🎉 all green!" if done == TOTAL else "keep going, re-run the cells you missed"))

This day needs FastAPI and httpx (used by FastAPI's test client). The next cell installs them if they're missing. You may need to restart the kernel afterwards.

In [ ]:
%pip install -q fastapi httpx
import logging
import os
import tempfile
import time
from dataclasses import dataclass
from pathlib import Path

import pytest
from fastapi import Depends, FastAPI, Header, HTTPException, Query, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

## Exercise 1: Depends: inject a repository and override it in tests  `medium`

Add two routes to `app1` that get their storage through `Depends(get_repo)`, never by using `_default_repo` directly:
- `GET /tasks` -> the list of tasks
- `POST /tasks` with body `TaskIn` -> the new task, status **201**

The check swaps in a fresh repository with `app1.dependency_overrides[get_repo]`, which is how you test FastAPI apps without a real database.

<details><summary>Hint</summary>

`def create_task(body: TaskIn, repo: TaskRepo = Depends(get_repo)):`. Put `status_code=201` on the decorator: `@app1.post("/tasks", status_code=201)`.

</details>

In [ ]:
class TaskRepo:
    def __init__(self):
        self.tasks = []

    def list(self):
        return list(self.tasks)

    def add(self, title):
        task = {"id": len(self.tasks) + 1, "title": title}
        self.tasks.append(task)
        return task


_default_repo = TaskRepo()

def get_repo() -> TaskRepo:
    return _default_repo


class TaskIn(BaseModel):
    title: str = Field(min_length=1)


app1 = FastAPI()

# your routes here

In [ ]:
def _test():
    fake = TaskRepo()
    app1.dependency_overrides[get_repo] = lambda: fake
    try:
        client = TestClient(app1)
        r = client.post("/tasks", json={"title": "Write tests"})
        assert r.status_code == 201, f"POST /tasks should return 201, got {r.status_code}"
        assert r.json() == {"id": 1, "title": "Write tests"}
        assert client.get("/tasks").json() == [{"id": 1, "title": "Write tests"}]
        assert fake.tasks and not _default_repo.tasks, "use Depends(get_repo) so the test's override is used"
        assert client.post("/tasks", json={"title": ""}).status_code == 422, "an empty title should fail validation"
    finally:
        app1.dependency_overrides.clear()

check('1. Depends: inject a repository and override it in tests', _test)

## Exercise 2: Auth: 401 vs 403 with dependencies  `interview`

`TOKENS` maps bearer tokens to users. Write:
- `get_current_user(authorization: str | None = Header(default=None))`: expects `"Bearer <token>"`. Missing, malformed or unknown -> `HTTPException(401)` with the header `WWW-Authenticate: Bearer`. Otherwise return the user dict.
- `require_admin(user = Depends(get_current_user))`: raises `HTTPException(403)` unless `user["role"] == "admin"`.
- `GET /me` -> `{"name": ...}` for any logged-in user.
- `DELETE /tasks/{task_id}` -> `{"deleted": task_id}`, admins only.

In a real app the token is a JWT you verify; the status-code logic stays the same.

<details><summary>Hint</summary>

`scheme, _, token = (authorization or "").partition(" ")`. Chaining dependencies (`require_admin` depends on `get_current_user`) is how FastAPI builds layered auth.

</details>

In [ ]:
TOKENS = {
    "alice-token": {"name": "alice", "role": "admin"},
    "bob-token": {"name": "bob", "role": "user"},
}


def get_current_user(authorization: str | None = Header(default=None)):
    pass


def require_admin(user: dict = Depends(get_current_user)):
    pass


app2 = FastAPI()

# your routes here

In [ ]:
def _test():
    client = TestClient(app2)
    bob = {"Authorization": "Bearer bob-token"}
    alice = {"Authorization": "Bearer alice-token"}
    r = client.get("/me")
    assert r.status_code == 401, f"no token should give 401, got {r.status_code}"
    assert r.headers.get("www-authenticate") == "Bearer", "a 401 should include WWW-Authenticate: Bearer"
    assert client.get("/me", headers={"Authorization": "Bearer nope"}).status_code == 401, "an unknown token is 401"
    assert client.get("/me", headers={"Authorization": "bob-token"}).status_code == 401, "a missing 'Bearer ' prefix is 401"
    assert client.get("/me", headers=bob).json() == {"name": "bob"}
    assert client.delete("/tasks/7", headers=bob).status_code == 403, "logged in but not admin is 403, not 401"
    r = client.delete("/tasks/7", headers=alice)
    assert r.status_code == 200 and r.json() == {"deleted": 7}
    assert client.delete("/tasks/7").status_code == 401, "no token on an admin route is still 401"

check('2. Auth: 401 vs 403 with dependencies', _test)

## Exercise 3: Pagination and filtering with Query validation  `medium`

Write `GET /items` on `app3` with query parameters:
- `limit`: default 10, between 1 and 100 (outside that range -> 422)
- `offset`: default 0, at least 0
- `category`: optional; when given, keep only items in that category

Return `{"total": <count after filtering>, "limit": ..., "offset": ..., "items": [...]}`. `total` lets the client draw page numbers.

<details><summary>Hint</summary>

`limit: int = Query(10, ge=1, le=100)`. Filter first, then slice: `filtered[offset:offset + limit]`. Interview follow-up: for huge tables, cursor (keyset) pagination (`WHERE id > last_seen_id`) beats `OFFSET`.

</details>

In [ ]:
ITEMS = [{"id": i, "category": "books" if i % 2 else "games"} for i in range(1, 26)]

app3 = FastAPI()

# your route here

In [ ]:
def _test():
    client = TestClient(app3)
    r = client.get("/items")
    assert r.status_code == 200, f"GET /items should exist, got {r.status_code}"
    body = r.json()
    assert body["total"] == 25 and body["limit"] == 10 and body["offset"] == 0
    assert [it["id"] for it in body["items"]] == list(range(1, 11)), "the default page is the first 10 items"
    body = client.get("/items", params={"limit": 5, "offset": 20}).json()
    assert [it["id"] for it in body["items"]] == [21, 22, 23, 24, 25]
    body = client.get("/items", params={"category": "games", "limit": 3}).json()
    assert body["total"] == 12, "total counts the filtered items, not the page"
    assert [it["id"] for it in body["items"]] == [2, 4, 6]
    assert client.get("/items", params={"limit": 0}).status_code == 422, "limit must be at least 1"
    assert client.get("/items", params={"limit": 500}).status_code == 422, "limit must be at most 100"
    assert client.get("/items", params={"offset": -1}).status_code == 422, "offset can't be negative"

check('3. Pagination and filtering with Query validation', _test)

## Exercise 4: Consistent errors and a timing middleware  `medium`

On `app4`:
1. Register an exception handler for `NotFoundError` that returns status **404** with JSON `{"error": {"code": "not_found", "message": "<resource> <id> not found"}}`. Routes then just `raise NotFoundError("task", 5)`, with no response-building in each route.
2. Add an HTTP middleware that sets the response header `X-Process-Time-Ms` (milliseconds, as a string) on **every** response, including errors.
3. `GET /tasks/{task_id}` returns `{"id": ..., "title": ...}` from `TASKS`, or raises `NotFoundError`.

<details><summary>Hint</summary>

`@app4.exception_handler(NotFoundError)` on an `async def handler(request, exc)` that returns `JSONResponse(...)`. `@app4.middleware("http")` on `async def timing(request, call_next)`: `response = await call_next(request)`, then set `response.headers[...]`.

</details>

In [ ]:
class NotFoundError(Exception):
    def __init__(self, resource, id):
        self.resource = resource
        self.id = id
        super().__init__(f"{resource} {id} not found")


TASKS = {1: "Write tests", 2: "Ship it"}

app4 = FastAPI()

# your handler, middleware and route here

In [ ]:
def _test():
    client = TestClient(app4)
    r = client.get("/tasks/1")
    assert r.status_code == 200 and r.json() == {"id": 1, "title": "Write tests"}, "GET /tasks/1 should return the task"
    assert "x-process-time-ms" in r.headers, "add X-Process-Time-Ms in an HTTP middleware"
    float(r.headers["x-process-time-ms"])
    r = client.get("/tasks/99")
    assert r.status_code == 404, f"a missing task should be 404, got {r.status_code}"
    assert r.json() == {"error": {"code": "not_found", "message": "task 99 not found"}}, "use the exception handler's error shape"
    assert "x-process-time-ms" in r.headers, "the middleware should time error responses too"

check('4. Consistent errors and a timing middleware', _test)

## Exercise 5: pytest: fixtures, parametrize, raises, monkeypatch  `medium`

`PRICING_CODE` below is a small module, `pricing.py`. Write a pytest file **as a string** in `TEST_CODE` that imports from `pricing` and includes:
- a `@pytest.fixture`
- a `@pytest.mark.parametrize` test with at least 3 cases for `parse_price`
- a `pytest.raises(ValueError)` test
- a test that uses `monkeypatch.setenv("TAX_RATE", ...)` to check `price_with_tax`

The check writes both files to a temp folder and runs real pytest on them. Every test must pass, and there must be at least 5 passing test cases.

<details><summary>Hint</summary>

Start with `TEST_CODE = '''import pytest\nfrom pricing import parse_price, price_with_tax\n...'''`. Each parametrize case counts as one test. `monkeypatch` changes are undone after each test, so tests don't leak state into each other.

</details>

In [ ]:
PRICING_CODE = '''
import os

def parse_price(text):
    """'$1,234.50' -> 1234.5. Raises ValueError for empty or negative input."""
    cleaned = text.strip().replace("$", "").replace(",", "")
    if not cleaned:
        raise ValueError("empty price")
    value = float(cleaned)
    if value < 0:
        raise ValueError("negative price")
    return value

def tax_rate():
    return float(os.environ.get("TAX_RATE", "0.18"))

def price_with_tax(text):
    return round(parse_price(text) * (1 + tax_rate()), 2)
'''

TEST_CODE = '''
# your pytest file here
'''

In [ ]:
def _test():
    for needed in ["@pytest.fixture", "@pytest.mark.parametrize", "pytest.raises", "monkeypatch"]:
        assert needed in TEST_CODE, f"TEST_CODE should use {needed}"

    class _Collect:
        passed = 0
        failed = []
        def pytest_runtest_logreport(self, report):
            if report.when == "call" and report.passed:
                self.passed += 1
            elif report.failed:
                self.failed.append(report.nodeid)

    with tempfile.TemporaryDirectory() as tmp:
        Path(tmp, "pricing.py").write_text(PRICING_CODE)
        Path(tmp, "test_pricing.py").write_text(TEST_CODE)
        collector = _Collect()
        code = pytest.main([tmp, "-q", "-p", "no:cacheprovider"], plugins=[collector])
    assert not collector.failed, f"failing tests: {collector.failed} (see the pytest output above)"
    assert code == 0, f"pytest exit code {int(code)} (5 means no tests were found)"
    assert collector.passed >= 5, f"only {collector.passed} test cases passed, write at least 5"

check('5. pytest: fixtures, parametrize, raises, monkeypatch', _test)

## Exercise 6: Logging levels and settings from the environment  `easy`

1. `charge(amount, logger)`: if `amount <= 0`, log an **error** `"invalid amount: <amount>"` and return `False`. If `amount > 10_000`, log a **warning** `"large charge: <amount>"`. Then log **info** `"charged <amount>"` and return `True`. Use the `logger`, not `print`.
2. `load_settings(env)` -> a frozen `Settings` dataclass, read from the `env` dict (in production you'd pass `os.environ`):
   - `APP_NAME` (default `"tasks"`), `DEBUG` (true for `"1"`, `"true"`, `"yes"`, any case; default false), `PAGE_SIZE` (int, default 20)
   - `DATABASE_URL` is **required**: raise `RuntimeError("DATABASE_URL is required")` if it's missing

<details><summary>Hint</summary>

Use `logger.error("invalid amount: %s", amount)`: passing arguments separately means the string is only formatted if the message is actually emitted. Settings come from the environment (12-factor), never hard-coded secrets.

</details>

In [ ]:
def charge(amount, logger):
    pass


@dataclass(frozen=True)
class Settings:
    app_name: str
    debug: bool
    page_size: int
    database_url: str


def load_settings(env):
    pass

In [ ]:
def _test():
    records = []

    class ListHandler(logging.Handler):
        def emit(self, record):
            records.append((record.levelname, record.getMessage()))

    log = logging.getLogger("day10.charge")
    log.handlers = [ListHandler()]
    log.setLevel(logging.DEBUG)
    log.propagate = False

    assert charge(0, log) is False, "an amount <= 0 should return False"
    assert records == [("ERROR", "invalid amount: 0")], "log an ERROR for an invalid amount (and nothing else)"
    records.clear()
    assert charge(50_000, log) is True
    assert records == [("WARNING", "large charge: 50000"), ("INFO", "charged 50000")], "warn on large charges, then log info"
    records.clear()
    charge(20, log)
    assert records == [("INFO", "charged 20")]

    s = load_settings({"DATABASE_URL": "postgresql://db/tasks"})
    assert s == Settings("tasks", False, 20, "postgresql://db/tasks"), "check the defaults"
    s = load_settings({"DATABASE_URL": "x", "DEBUG": "TRUE", "PAGE_SIZE": "50", "APP_NAME": "api"})
    assert s.debug is True and s.page_size == 50 and s.app_name == "api", "parse DEBUG case-insensitively and PAGE_SIZE as int"
    assert raises(RuntimeError, load_settings, {}), "a missing DATABASE_URL should raise RuntimeError"
    assert raises(AttributeError, setattr, s, "debug", False), "Settings should be frozen (keep @dataclass(frozen=True))"

check('6. Logging levels and settings from the environment', _test)

## Exercise 7: Interview: an idempotent POST /payments  `interview`

Mobile clients retry when the network drops, and a retry must never **charge twice**. Write `POST /payments` on `app7`:
- the client sends an `Idempotency-Key` header. If it's missing, return **400**
- body: `PaymentIn` (`amount > 0`, 3-letter `currency`)
- **first time** a key is seen: append the charge to `CHARGES`, store the response in `IDEMPOTENCY_STORE[key]`, and return **201** with `{"payment_id": n, "amount": ..., "currency": ...}` (`n` = number of charges so far)
- **same key, same body**: return the stored response (201) and don't charge again
- **same key, different body**: return **409** (the client reused a key by mistake)

Use the `CHARGES` list and `IDEMPOTENCY_STORE` dict provided (the check resets them).

<details><summary>Hint</summary>

`idempotency_key: str | None = Header(default=None)` reads the `Idempotency-Key` header (FastAPI turns `_` into `-`). Store `(body.model_dump(), response)` for each key. In production this is a DB table with a **unique constraint** on the key, so two concurrent retries can't both insert, plus an expiry time.

</details>

In [ ]:
CHARGES = []
IDEMPOTENCY_STORE = {}


class PaymentIn(BaseModel):
    amount: float = Field(gt=0)
    currency: str = Field(min_length=3, max_length=3)


app7 = FastAPI()

# your route here

In [ ]:
def _test():
    CHARGES.clear()
    IDEMPOTENCY_STORE.clear()
    client = TestClient(app7)
    body = {"amount": 499.0, "currency": "INR"}
    r1 = client.post("/payments", json=body, headers={"Idempotency-Key": "k1"})
    assert r1.status_code == 201, f"the first request should return 201, got {r1.status_code}"
    assert r1.json() == {"payment_id": 1, "amount": 499.0, "currency": "INR"}
    r2 = client.post("/payments", json=body, headers={"Idempotency-Key": "k1"})
    assert r2.status_code == 201 and r2.json() == r1.json(), "a retry with the same key returns the same response"
    assert len(CHARGES) == 1, "a retry must NOT charge again"
    r3 = client.post("/payments", json=body, headers={"Idempotency-Key": "k2"})
    assert r3.json()["payment_id"] == 2 and len(CHARGES) == 2, "a new key is a new payment"
    r4 = client.post("/payments", json={"amount": 1.0, "currency": "INR"}, headers={"Idempotency-Key": "k1"})
    assert r4.status_code == 409, "the same key with a different body should return 409"
    assert client.post("/payments", json=body).status_code == 400, "a missing Idempotency-Key should return 400"
    assert client.post("/payments", json={"amount": -5, "currency": "INR"}, headers={"Idempotency-Key": "k9"}).status_code == 422
    assert len(CHARGES) == 2

check('7. Interview: an idempotent POST /payments', _test)

## Your score

In [ ]:
score()

## Say it out loud (interview questions)

Answer each one aloud in under 60 seconds, as if a client asked. Model answers are in `solutions.ipynb`.

- Walk me through how you'd structure a FastAPI service with 20 endpoints.
- Users report that `GET /orders` got slow after a release. How do you find the cause?
- You need to make a breaking change to an API that mobile apps use. How do you do it?
- In this service, what would you unit-test and what would you integration-test?
- Where do database passwords and API keys live, and how does the app get them?